In [1]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.DataFrame
import java.time.{LocalDateTime, Duration, Period}
import java.time.temporal.ChronoUnit

Intitializing Scala interpreter ...

Spark Web UI available at http://c8fbbb957f25:4040
SparkContext available as 'sc' (version = 3.5.8, master = local[*], app id = local-1779284613799)
SparkSession available as 'spark'

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.DataFrame
import java.time.{LocalDateTime, Duration, Period}
import java.time.temporal.ChronoUnit

# Caricamento e preparazione dati

In [2]:
// VendorID, pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, trip_distance, fare_amount, tip_amount, total_amount
val tripRdd = spark.read
    .parquet("/workspace/data/sample.parquet")
    .drop("store_and_fwd_flag", "RatecodeID", "passenger_count", "extra", "mta_tax", "tolls_amount", "ehail_fee", "improvement_surcharge", "payment_type", "trip_type", "congestion_surcharge", "cbd_congestion_fee", "Airport_fee", "VendorID")
    .rdd
    .map(row => (
        row.getAs[LocalDateTime](0), // pickup_datetime
        row.getAs[LocalDateTime](1), // dropoff_datetime
        row.getDouble(2), // trip_distance
        row.getLong(3), // PULocationID
        row.getLong(4), // DOLocationID
        row.getDouble(5), // fare_amount
        row.getDouble(6), // tip_amount
        row.getDouble(7) // total_amount
    ))
// LocationID, Borough
val zonesRdd = spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/workspace/data/taxi_zone_lookup.csv")
    .drop("service_zone", "Zone")
    .rdd
    .map[(Long, String)](row => (
        row.getInt(0), // LocationID
        row.getString(1) // Borough
    ))

tripRdd: org.apache.spark.rdd.RDD[(java.time.LocalDateTime, java.time.LocalDateTime, Double, Long, Long, Double, Double, Double)] = MapPartitionsRDD[8] at map at <console>:32
zonesRdd: org.apache.spark.rdd.RDD[(Long, String)] = MapPartitionsRDD[24] at map at <console>:49

In [3]:
implicit val localDateTimeOrdering: Ordering[LocalDateTime] = Ordering.fromLessThan(_ isBefore _)
val tripDepTimes = tripRdd.map(_._1)
val tripDataMinDate = tripDepTimes.min()
val tripDataMaxDate = tripDepTimes.max()

localDateTimeOrdering: Ordering[java.time.LocalDateTime] = scala.math.Ordering$$anon$4@1ffa00c2
tripDepTimes: org.apache.spark.rdd.RDD[java.time.LocalDateTime] = MapPartitionsRDD[25] at map at <console>:29
tripDataMinDate: java.time.LocalDateTime = 2009-01-01T00:02:59
tripDataMaxDate: java.time.LocalDateTime = 2020-07-01T00:02:40

# Jobs

## Standard

### 1. Numero di corse per quartiere in un giorno

In [4]:
val daysDelta = ChronoUnit.DAYS.between(tripDataMinDate, tripDataMaxDate)

daysDelta: Long = 4198

In [5]:
tripRdd.map(t => (t._4, 1))
    .join(zonesRdd)
    .map(t => (t._2._2, 1))
    .reduceByKey(_ + _)
    .mapValues(_ / daysDelta)
    .collect

res0: Array[(String, Long)] = Array((Staten Island,0), (Queens,8), (N/A,0), (EWR,0), (Manhattan,114), (Brooklyn,4), (Bronx,2), (Unknown,1))

### 2. Numero medio di corse per fasce orario in un giorno

In [6]:
tripRdd
    .map(t => (t._1.getHour, 1))
    .reduceByKey(_ + _)
    .mapValues(_ / daysDelta)
    .collect

res1: Array[(Int, Long)] = Array((15,9), (21,3), (0,1), (6,3), (3,1), (12,9), (9,6), (18,8), (13,9), (19,6), (4,1), (16,9), (22,3), (1,1), (7,4), (10,7), (11,8), (14,9), (23,2), (17,9), (8,5), (20,4), (5,1), (2,1))

### 3. Numero medio di corse per ciascun mese dell'anno

In [7]:
val yearsDelta = ChronoUnit.YEARS.between(tripDataMinDate, tripDataMaxDate)
tripRdd
    .map(t => (t._1.getMonth.getValue, 1))
    .reduceByKey(_ + _)
    .mapValues(_ / yearsDelta)
    .collect

yearsDelta: Long = 11
res2: Array[(Int, Long)] = Array((6,49980), (1,0), (7,0), (5,0))

### 4. Velcità media per fascia oraria

In [8]:
tripRdd
    .map(t => (t._1.getHour, (t._3, ChronoUnit.SECONDS.between(t._1, t._2).doubleValue / 3600.doubleValue)))
    .reduceByKey((v1, v2) => (v1._1 + v2._1, v1._2 + v2._2))
    .mapValues(v => v._1 / v._2)
    .collect

res3: Array[(Int, Double)] = Array((15,11.760140306200412), (21,16.815156678194118), (0,18.600984331725737), (6,14.418934719797903), (3,74.39415888688), (12,35.79148462116623), (9,12.540451098918172), (18,13.778967492056141), (13,11.239316806057184), (19,14.984218250866967), (4,-13.328894692431849), (16,14.09333902324226), (22,18.871361890332622), (1,20.510818501439463), (7,16.63224652430091), (10,12.578709205747515), (11,11.894081118601392), (14,16.952366359240884), (23,18.711613784164207), (17,12.871267669288734), (8,13.025937844498342), (20,15.77367549222801), (5,25.289904353177604), (2,21.45997603092267))

### 5. Mancia media in base al quartiere di destinazione

In [9]:
tripRdd
    .map(t => (t._5, (t._7, 1)))
    .join(zonesRdd)
    .map(t => (t._2._2, t._2._1))
    .reduceByKey((v1, v2) => (v1._1 + v2._1, v1._2 + v2._2))
    .mapValues(v => v._1 / v._2)
    .collect

res4: Array[(String, Double)] = Array((Staten Island,1.3670460469157255), (Queens,2.05602432465469), (N/A,5.98982039091389), (EWR,10.209050279329619), (Manhattan,1.685329059838481), (Brooklyn,2.4735022538689058), (Bronx,1.315107757506847), (Unknown,1.5186058700209641))

## Optimized

In [10]:
// Broadcast the zones lookup table to avoid joins
val broadcastZones = spark.sparkContext.broadcast(zonesRdd.collectAsMap())

broadcastZones: org.apache.spark.broadcast.Broadcast[scala.collection.Map[Long,String]] = Broadcast(24)

### 1. Numero di corse per quartiere in un giorno

Ottimizzato utilizzando broadcast + flatMap invece di join

In [11]:
tripRdd
    .flatMap(t => broadcastZones.value.get(t._4).map(borough => (borough, 1)))
    .reduceByKey(_ + _)
    .mapValues(_ / daysDelta)
    .collect

res5: Array[(String, Long)] = Array((Staten Island,0), (Queens,8), (N/A,0), (EWR,0), (Manhattan,114), (Brooklyn,4), (Bronx,2), (Unknown,1))

### 2+4. Numero medio di corse per fasce orario in un giorno / Velcità media per fascia oraria

Ottimizzati unendo in 2 job in un'unica chiamata di reduce dato che vengono "ridotti" per la stessa chiave

In [12]:
tripRdd
    .map(t => (t._1.getHour, (1, t._3,ChronoUnit.SECONDS.between(t._1, t._2).doubleValue / 3600.doubleValue)))
    .reduceByKey((v1, v2) => (v1._1 + v2._1, v1._2 + v2._2, v1._3 + v2._3))
    .map(v => (v._1, v._2._1 / daysDelta, v._2._2 / v._2._3))
    .collect

res6: Array[(Int, Long, Double)] = Array((15,9,11.760140306200412), (21,3,16.815156678194118), (0,1,18.600984331725737), (6,3,14.418934719797903), (3,1,74.39415888688), (12,9,35.79148462116623), (9,6,12.540451098918172), (18,8,13.778967492056141), (13,9,11.239316806057184), (19,6,14.984218250866967), (4,1,-13.328894692431849), (16,9,14.09333902324226), (22,3,18.871361890332622), (1,1,20.510818501439463), (7,4,16.63224652430091), (10,7,12.578709205747515), (11,8,11.894081118601392), (14,9,16.952366359240884), (23,2,18.711613784164207), (17,9,12.871267669288734), (8,5,13.025937844498342), (20,4,15.77367549222801), (5,1,25.289904353177604), (2,1,21.45997603092267))

### 3. Numero medio di corse per ciascun mese dell'anno

In [13]:
tripRdd
    .map(t => (t._1.getMonth.toString, 1))
    .reduceByKey(_ + _)
    .mapValues(_ / yearsDelta)
    .collect

res7: Array[(String, Long)] = Array((JUNE,49980), (MAY,0), (JANUARY,0), (JULY,0))

### 5. Mancia media in base al quartiere di destinazione

Ottimizzato utilizzando broadcast + flatMap invece di join


In [ ]:
tripRdd
    .flatMap(t => broadcastZones.value.get(t._5).map(borough => (borough, (t._7, 1))))
    .reduceByKey((v1, v2) => (v1._1 + v2._1, v1._2 + v2._2))
    .mapValues(v => v._1 / v._2.toDouble)
    .collect

res9: Array[(String, Double)] = Array((Staten Island,1.3670460469157253), (Queens,2.056024324654701), (N/A,5.98982039091389), (EWR,10.209050279329619), (Manhattan,1.685329059838206), (Brooklyn,2.473502253868916), (Bronx,1.3151077575068482), (Unknown,1.5186058700209641))